In [ ]:
-- ============================================================
-- Unity Catalog Setup & Governance
-- Run this in the Databricks SQL Editor before executing any
-- notebooks. It creates the three-level namespace and applies
-- governance metadata (comments, tags) to key tables.
-- ============================================================

-- 1. Create a dedicated catalog for the project
CREATE CATALOG IF NOT EXISTS fpl_project;

-- 2. One schema per medallion layer
CREATE SCHEMA IF NOT EXISTS fpl_project.bronze;
CREATE SCHEMA IF NOT EXISTS fpl_project.silver;
CREATE SCHEMA IF NOT EXISTS fpl_project.gold;

-- 3. Volume for landing raw JSON files from the FPL API
CREATE VOLUME IF NOT EXISTS fpl_project.bronze.raw_files;

-- 4. Governance: add a human-readable description to the gold table
COMMENT ON TABLE fpl_project.gold.player_value_rankings
    IS 'Player value rankings: points per million, per 90, with xG/xA stats';

-- 5. Governance: tag the table for discoverability and lineage
ALTER TABLE fpl_project.gold.player_value_rankings
    SET TAGS ('domain' = 'fantasy_football', 'layer' = 'gold', 'refresh' = 'weekly');

-- 6. Column-level documentation
ALTER TABLE fpl_project.silver.players
    ALTER COLUMN ownership_pct
    COMMENT 'Percentage of all FPL managers who own this player';

-- 7. Delta time travel: inspect the version history of a table
DESCRIBE HISTORY fpl_project.silver.players;